# mRFP: nucleotide representations with a constant protein

This notebook asks whether nucleotide encoder representations help predict expression beyond selected sequence-composition and predicted RNA-folding features. It extends Stage 1's control using monomeric red fluorescent protein (mRFP): the coding sequences vary while the amino-acid sequence they encode stays the same. This holds protein sequence constant while we examine nucleotide variation.

Each row in `mRFP_Expression.csv` pairs a coding sequence with a supplied expression score. The original study expressed variants of the mRFP gene in *E. coli* and measured red fluorescence as a proxy for protein production, also checking its relationship to protein abundance using proteomics. The supplied `Value` is the natural log of mean red fluorescence after normalization using a green GFPuv reference. It measures reporter production on a log fluorescence scale, not in molecule counts. The [target guide](../../docs/methods/data-and-targets.md#mrfp-fluorescence-to-the-value-column) traces the measurements, formula and matching CSV values. [Nieuwkoop et al., 2023](https://academic.oup.com/nar/article/51/5/2363/7016452)

Synonymous codons specify the same amino acid: coding-DNA triplets `GCT` and `GCC`, for example, correspond to RNA codons `GCU` and `GCC`, both encoding alanine. Such differences between rows can preserve the entire protein sequence. Protein encoders then receive identical amino-acid input, giving a negative control that cannot distinguish the variants from its sequence input. DNA and RNA encoders can receive different nucleotide sequences between variants. Converting T to U creates the RNA view of each row; it does not create the synonymous differences between rows.

We reuse Stage 1's saved, pooled embeddings and first check the constant-protein premise. We then construct named sequence controls, compare separate ridge predictors, and ask whether embeddings improve prediction after the controls have been used. The pretrained encoders stay frozen. This establishes a boundary case for the fusion investigation: an additional protein representation receives no sequence differences with which to distinguish these variants, while nucleotide representations may make useful variation accessible to a predictor. The notebook evaluates that representation usefulness; it does not compare fusion architectures or establish that combining the best nucleotide encoder with another encoder will help.

**Project context:** research-plan item 5; [Gantt and Details task 18](../../docs/Research_Project_Plan_Gantt.xlsx); Overleaf Section 12.2, item 5. The [Stage 1 overview](README.md#mrfp-changing-the-nucleotide-sequence-while-keeping-the-protein-fixed) connects this control to the encoder-swap experiments.


---

## Setup

Prepare the Python environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies), including ViennaRNA for folding calculations. Select that environment as this notebook's kernel. Package installation belongs in that setup step; the cells below do not install packages.

### Imports

In [3]:
# Files and project paths
import os
import sys
from pathlib import Path

# Numerical arrays (NumPy) and result tables (pandas).
import numpy as np
import pandas as pd

# ViennaRNA is needed only for an explicitly enabled folding calculation.
try:
    import RNA
except ModuleNotFoundError as error:
    if error.name != "RNA":
        raise
    RNA = None

# Compare feature-expression rankings and score held-out predictions.
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

# Regression, feature scaling and cross-validation for the predictors.
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Table display width in characters, unrelated to nucleotide-window lengths.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("ViennaRNA", RNA.__version__ if RNA is not None else "not installed (reading saved results is available)")

ViennaRNA 2.7.2


### Project paths and shared code

Locate the project's `Code` directory and import the shared helpers used to load data, read saved embeddings and evaluate representations. Later input and result paths are anchored to this directory.

In [4]:
# Search the kernel's working folder and its ancestors for either layout:
# Code/shared_code from the repository root, or shared_code from inside Code.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    # A for-loop's else runs only when no match ends the search with break.
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Let Python find shared_code without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Reuse project helpers for data, saved embeddings, sequence features and analysis.
from shared_code import analysis, datasets, embeddings, mrfp_cache, sequences

# Encoder settings and folds that keep identical sequences together.
from shared_code.encoders import ENCODERS
from shared_code.splits import grouped_folds

### Settings and file locations

Reuse Stage 1's 1,000-row sample, selected from the supplied dataset of 1,459 rows in the **Pipeline control: synonymous recoding** section of [Stage 1's stability notebook](Stage1_stability.ipynb). Sampling seed 42 reproduces that sample and identifies its matching saved embeddings. Fold-assignment seeds divide the **1,000 sampled rows** into five folds for evaluation. Different assignments rearrange their fold membership; they do not select another sample from the full dataset.

The default `READ_SAVED_RESULTS = True` reads the existing result tables and reports missing work. It does not load embeddings, calculate sequence controls, fit predictors, or write files. The sample and fold settings below describe the intended experiment; read mode checks the requested result names and assignment seeds, without claiming that historical producers used today's settings.

The folding windows and result locations stay unchanged. Explicit calculation mode retains strict input and settings checks; legacy tables without that evidence cannot be reused as verified caches or overwritten.


In [5]:
# Default Run All inspects historical tables; it does not fill gaps or write results.
READ_SAVED_RESULTS = True

# Reproduce the Stage 1 sample associated with the saved embedding directory.
CONTROL = "mrfp_expression"
N_ROWS = 1000
SEED = 42

# Compare Stage 1's pinned partition with assignments using seeds 0 through 9.
PROBE_SEED = 42
N_ASSIGNMENTS = 10

# Prefix lengths in nucleotides, measured from the coding sequence's start.
TRACK_A_WINDOW = 200  # Nussinov pairing fraction, inherited from Stage 1.
START_WINDOW = 60     # ViennaRNA folding energy near the start codon.

# Read Stage 1 inputs and keep this notebook's result tables in Stage 4.
EMBEDDING_DIR = (
    CODE_DIR / "stage1" / "embeddings" / "mrfp_expression_n1000_seed42"
)
RESULTS_DIR = CODE_DIR / "stage4" / "results" / "tier1_mrfp"
STRUCTURE_CACHE = RESULTS_DIR / "structure_features.csv"
PROBE_CACHE = RESULTS_DIR / "probe_scores.csv"
GAIN_CACHE = RESULTS_DIR / "gains_over_controls.csv"

### Read existing tables or prepare an authorized calculation

Read mode inspects the three CSVs independently. It checks required columns, finite scores, integer seed keys and duplicate entries. An incomplete model or control/model comparison is reported and excluded from its summary; other complete comparisons remain readable. Structure rows are shown as historical features without reconstructing their alignment to today's sample. Missing files are reported, and no replacement is created.

Historical rows without a calculation fingerprint are labelled **unknown identity**. A recorded fingerprint is also not revalidated against inputs in read mode. These tables describe saved work; their presence does not establish the producer settings or scientific correctness. This path needs neither the Stage 1 embedding files nor ViennaRNA. Earlier diagnostic outputs retained in the notebook remain historical; read mode does not regenerate them.

Setting `READ_SAVED_RESULTS = False` enables the existing calculation path only for an authorized run. That path verifies exact keys and calculation fingerprints before reusing results, and computes missing compatible entries. It refuses unidentified or conflicting history; a separately authorized recalculation needs a separate result destination. It never adds today's identity to old results.


In [ ]:
def save(frame, path):
    """Write a table only in the explicitly enabled calculation path."""
    if READ_SAVED_RESULTS:
        raise RuntimeError("Saved-results mode cannot write result tables.")
    # Finish the new CSV before replacing the previous complete table.
    partial = path.with_name(path.name + ".partial")
    frame.to_csv(partial, index=False)
    os.replace(partial, path)




In [6]:
CACHE_FIELDS = {
    STRUCTURE_CACHE: (("sequence",), ("nussinov_200", "mfe_start_60", "mfe_full")),
    PROBE_CACHE: (("model", "seed"), ("seed", "r2")),
    GAIN_CACHE: (("controls", "model", "seed"), ("seed", "controls R2", "with encoder R2")),
}
if READ_SAVED_RESULTS:
    # Inspect each file independently; missing or ambiguous rows are not repaired.
    SAVED_TABLES = {
        path: mrfp_cache.read_saved(path, keys=keys, numeric=numeric)
        for path, (keys, numeric) in CACHE_FIELDS.items()
    }
    print("Read saved results only: historical identities are not verified against today's inputs.")
    for table in SAVED_TABLES.values():
        for issue in table.issues:
            print(issue)
else:
    if RNA is None:
        raise ImportError("Calculation mode requires ViennaRNA; see the root README setup instructions.")
    # Strict calculation reuse never certifies or replaces unidentified history.
    for path, (keys, numeric) in CACHE_FIELDS.items():
        if path.exists():
            mrfp_cache.checked_rows(
                pd.read_csv(path).to_dict("records"),
                keys=keys, expected={}, numeric=numeric, source=path,
            )
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)


### Load the sample and saved embeddings

Load the coding sequences and their already-transformed `Value` scores, then read their pooled embeddings from Stage 1. `y` retains the natural log of GFPuv-normalized mean red fluorescence; the loader and the float conversion below do not apply another logarithm.

**Why normalize against GFPuv?** The mRFP coding sequence changes between variants, while the separately expressed GFPuv construct stays the same on the same plasmid. Its green signal provides an internal reference. Dividing red by green can reduce variation that affects both reporters proportionally; multiplying by the dataset's average green signal puts each red measurement at a common reference level. The study used weak GFPuv expression to limit interference with mRFP production. [Original study, Plasmid preparation and library generation](https://academic.oup.com/nar/article/51/5/2363/7016452)

**Illustrative example.** Suppose the average green signal across a hypothetical dataset is 100. The three observations below illustrate that reference; they are not the entire hypothetical dataset or measured rows from our CSV.

| Observation | Red signal | Green signal | Corrected red: red $\times$ 100 / green |
| --- | ---: | ---: | ---: |
| A | 100 | 50 | 200 |
| B | 200 | 100 | 200 |
| C | 200 | 50 | 400 |

B is twice as bright as A in both channels, so their corrected red signals agree. C has twice A's red signal without a green increase, so its corrected value remains twice as large. This depends on the reference responding appropriately: effects that influence only one reporter do not cancel, and green-signal noise or competition between reporters can affect the ratio.

**Why take the log?** Equal fold changes become equal differences in `y`: doubling corrected fluorescence from 100 to 200 or from 10,000 to 20,000 adds the same $\ln(2)\approx0.693$. This changes how prediction errors are weighted, as the scoring section below explains. Natural log is a choice of units; log base 2 would express a doubling as 1. The preparation code confirms `np.log`, but does not document the authors' reason for choosing it. Logging does not guarantee normally distributed errors or remove biological confounding. The [target guide](../../docs/methods/data-and-targets.md#gfpuv-normalization-and-the-natural-logarithm) gives the full derivation, limits and a real source-to-CSV example.

Each saved embedding is one numerical vector per sequence. All feature matrices must describe the same rows in the same order; the saved-input checks compare their identity with the requested sequences. This cell reads saved representations without loading or running pretrained encoders. Missing or incompatible embeddings must be resolved with the Stage 1 producer. The output checks the sample, sequence lengths, expression range and matrix widths before any prediction is evaluated.

**Read mode:** the cell below retains only the encoder labels for checking saved result coverage. Sample and embedding loading are skipped, so missing Stage 1 matrices do not prevent inspection of historical score tables.


In [7]:
# The same encoder labels define expected saved comparisons in either mode.
MODALITY = {e.key: e.modality for e in ENCODERS.values()}
LABEL = {e.key: e.label for e in ENCODERS.values()}

if READ_SAVED_RESULTS:
    print("Dataset and embedding loading skipped; historical score tables do not require them.")
else:
    # Keep sequences, expression targets and split groups in the loader's row order.
    control = datasets.load_dataset(CONTROL, n_rows=N_ROWS, seed=SEED)
    seqs = control.sequences
    y = control.labels.astype(float)
    groups = control.groups  # Identical sequences must remain in the same fold.

    EMBEDDINGS = {
        key: embeddings.read_saved(encoder, seqs, EMBEDDING_DIR)
        for key, encoder in ENCODERS.items()
    }

    # Check the sample and cached matrix widths before constructing comparisons.
    print(
        f"{len(seqs)} rows ({control.n_sampled} sampled), {len(set(seqs))} distinct sequences, "
        f"lengths {sorted({len(s) for s in seqs})}, expression {y.min():.2f} to {y.max():.2f}"
    )
    print(
        f"{len(EMBEDDINGS)} cached embeddings read: "
        + ", ".join(f"{k} {v.shape[1]}" for k, v in EMBEDDINGS.items())
    )

    # Bind later scores to the actual row order, groups and expression targets.
    sample_identity = mrfp_cache.fingerprint(
        {
            "dataset": CONTROL, "sample_rows": N_ROWS, "sample_seed": SEED,
            "sequences": list(seqs), "groups": [str(group) for group in groups],
        },
        arrays=(y,),
    )


1000 rows (1000 sampled), 997 distinct sequences, lengths [678], expression 7.62 to 11.38
12 cached embeddings read: nt500m 1280, ntv2_100m 512, dnabert2 768, hyenadna 256, rnafm 640, rinalmo 640, mrnafm 1280, calm 768, esm2_8m 320, esm2_35m 480, esm2_150m 640, protbert 1024


The displayed sample includes repeated nucleotide sequences, and all its coding sequences have the same length. `seqs`, `y` and `groups` keep the sequences, expression targets and evaluation groups aligned; `EMBEDDINGS` supplies the matching predictor inputs. Identical sequences will stay in the same validation fold. We can now check whether the nucleotide variants also preserve one translated protein.


---

## Check the constant-protein premise

Before predicting expression, translate the sampled nucleotide sequences and inspect how much their cached representations vary. The intended control has one amino-acid sequence shared by all variants: protein encoders receive identical input, while nucleotide encoders can receive different input.

For each encoder, the centroid is its average embedding. The table reports the mean Euclidean distance of its rows from that centroid. This describes variation within that representation; differently scaled embedding spaces cannot be ranked for predictive quality by these distances.


In [3]:
if READ_SAVED_RESULTS:
    print("Protein/embedding diagnostics skipped: no standalone saved table; retained outputs are historical.")
else:
    # Check that nucleotide variation preserves one translated protein sequence.
    proteins = {sequences.translate_cds(s) for s in seqs}

    # Measure each encoder's variation around its own average embedding.
    spread = pd.DataFrame(
        {
            "modality": [MODALITY[k] for k in EMBEDDINGS],
            "width": [EMBEDDINGS[k].shape[1] for k in EMBEDDINGS],
            "mean distance to centroid": [
                analysis.centroid_spread(EMBEDDINGS[k]) for k in EMBEDDINGS
            ],
        },
        index=[LABEL[k] for k in EMBEDDINGS],
    )
    print(f"distinct proteins: {len(proteins)}")
    display(spread.round(5))


distinct proteins: 1


,modality,width,mean distance to centroid
NT 500M human-ref,DNA,1280,4.72361
NT v2 100M multi-species,DNA,512,1.26535
DNABERT-2,DNA,768,0.74851
HyenaDNA large,DNA,256,0.61138
RNA-FM,RNA,640,0.81823
RiNALMo 150M,RNA,640,3.96281
mRNA-FM,RNA,1280,1.74951
CaLM,RNA,768,1.99879
ESM-2 8M,protein,320,0.00004
ESM-2 35M,protein,480,0.00004


The diagnostic reports one translated protein and very small spread in the protein embeddings, alongside variation in nucleotide embeddings. This is consistent with the intended synonymous control. Embedding spread alone does not establish cache provenance; the earlier saved-input checks provide that identity comparison.

The nucleotide representations vary, but this check does not say what that variation predicts. We next construct features with named sequence meanings so their predictive usefulness can be compared with the embeddings.


---

## Build sequence controls

The controls summarize properties of each nucleotide sequence. They give us a reference for asking whether embeddings improve expression prediction beyond these selected summaries. Amino-acid composition and sequence length are constant in this sample, so they are omitted.

RNA can form base pairs within its own sequence, creating secondary structure. Nussinov summarizes the maximum fraction of bases that can be paired under its pairing rules. ViennaRNA estimates minimum folding free energy, in kcal/mol, using a model of the energetic favourability of RNA secondary structures. This energy is calculated from the sequence, not obtained from the fluorescence measurements. Neither calculation measures these RNAs' structures inside cells.

| Control input | Meaning in this comparison |
| --- | --- |
| GC | Fraction of nucleotide positions occupied by G or C. |
| GC3 | Fraction of third codon positions occupied by G or C. |
| Codon use | Frequencies of the 64 codons, supplying one feature column per codon. |
| Nussinov, first 200 nt | Paired-base fraction in the retained Stage 1 prefix. |
| MFE, first 60 nt | ViennaRNA minimum folding free energy for the coding sequence's initial region. |
| MFE, whole sequence | The same energy calculation over the entire coding sequence. |

The 200-nucleotide Nussinov window and the 60-nucleotide ViennaRNA window both begin at the coding sequence's start. Their lengths retain the existing analysis choices; this notebook does not compare window lengths to establish an optimum.

### Calculate or reuse folding features

Folding produces the three structural summaries used alongside composition features. Read mode previews usable saved rows, without recalculating folding or certifying their coverage of the current sample. In calculation mode, the guarded calculation cell computes uncached sequences in batches, then restores sampled row order to align every result with expression scores and embeddings. That path requires a unique sequence key and the matching folding settings/source fingerprint before reuse; unidentified historical caches or a changed window stop before calculation.


In [ ]:
def structure_row(seq):
    # ViennaRNA uses RNA bases; convert the coding DNA view from T to U.
    rna = sequences.dna_to_rna(seq)

    # RNA.fold returns (structure, energy); [1] retains the energy in kcal/mol.
    return {
        "sequence": seq,
        "nussinov_200": sequences.nussinov_pairing_fraction(seq[:TRACK_A_WINDOW]),
        "mfe_start_60": RNA.fold(rna[:START_WINDOW])[1],
        "mfe_full": RNA.fold(rna)[1],
    }




In [ ]:
if READ_SAVED_RESULTS:
    structure_saved = SAVED_TABLES[STRUCTURE_CACHE]
    structure = pd.DataFrame(list(structure_saved.rows.values()))
    print(
        f"{len(structure)} usable saved structure rows; current sample coverage and folding settings "
        "are not verified in read mode. No folding is performed."
    )
    if not structure.empty:
        display(structure.head())


structure features: 997 computed now, 3 read back


In [ ]:
if not READ_SAVED_RESULTS:
    # Each sequence is its own input; the fingerprint identifies the folding rule.
    structure_identity = mrfp_cache.fingerprint(
        {
            "track_a_window": TRACK_A_WINDOW,
            "start_window": START_WINDOW,
            "viennarna": RNA.__version__,
        },
        functions=(
            structure_row, sequences.dna_to_rna,
            sequences.nussinov_pairing_fraction, sequences.can_pair,
        ),
    )
    structure_expected = {(seq,): structure_identity for seq in dict.fromkeys(seqs)}
    structure_keys, structure_numeric = CACHE_FIELDS[STRUCTURE_CACHE]
    done = pd.read_csv(STRUCTURE_CACHE) if STRUCTURE_CACHE.exists() else pd.DataFrame()
    saved = mrfp_cache.checked_rows(
        done.to_dict("records"), keys=structure_keys, expected=structure_expected,
        numeric=structure_numeric, source=STRUCTURE_CACHE,
    )

    # Calculate each missing sequence once, preserving first-occurrence order.
    todo = [seq for seq in dict.fromkeys(seqs) if (seq,) not in saved]
    for start in range(0, len(todo), 50):
        chunk = [
            {**structure_row(seq), mrfp_cache.IDENTITY_COLUMN: structure_identity}
            for seq in todo[start:start + 50]
        ]
        records = mrfp_cache.upsert(
            done.to_dict("records"), chunk,
            keys=structure_keys, expected=structure_expected,
            numeric=structure_numeric, source=STRUCTURE_CACHE,
        )
        done = pd.DataFrame(records)
        save(done, STRUCTURE_CACHE)

    # Restore sampled row order, including duplicates, to align with y and embeddings.
    structure = done.set_index("sequence").loc[seqs].reset_index(drop=True)
    # The 'read back' count includes repeated sample rows, not just cached sequences.
    print(f"structure features: {len(todo)} computed now, {len(seqs) - len(todo)} read back")


### Define the control inputs

Combine the folding results with GC, GC3 and codon frequencies. `CONTROLS` holds the matrix for each individual predictor. The scalar-feature table, `FEATURES`, is derived from those same matrices, so a control's calculation has one definition.


In [ ]:
if READ_SAVED_RESULTS:
    # Labels identify historical comparisons; feature matrices are not reconstructed.
    CONTROLS = dict.fromkeys((
        "GC", "GC3", "codon use (64)", "Nussinov, first 200 nt",
        "MFE, first 60 nt", "MFE, whole sequence",
    ))
    print("Sequence composition and folding feature construction skipped.")
else:
    # Skip GC, GC3 and log length in composition_features; take its 64 codon frequencies.
    codon_use = np.array([sequences.composition_features(s)[3:3 + 64] for s in seqs])

    # Define each control once as a matrix: sampled rows by feature columns.
    # Reshape scalar arrays, or select a one-column DataFrame, to retain two dimensions.
    CONTROLS = {
        "GC": np.array([sequences.gc_content(s) for s in seqs]).reshape(-1, 1),
        "GC3": np.array([sequences.gc3_content(s) for s in seqs]).reshape(-1, 1),
        "codon use (64)": codon_use,
        "Nussinov, first 200 nt": structure[["nussinov_200"]].to_numpy(),
        "MFE, first 60 nt": structure[["mfe_start_60"]].to_numpy(),
        "MFE, whole sequence": structure[["mfe_full"]].to_numpy(),
    }

    # Scalar summaries use the same definitions; the 64-column codon block is excluded.
    FEATURES = pd.DataFrame({
        name: values[:, 0]
        for name, values in CONTROLS.items()
        if values.shape[1] == 1
    })


### Group the controls and inspect their associations

The **Track A controls** combine GC3, codon frequencies and the Nussinov fraction. The expanded **all controls** group contains the Track A features plus GC content and both folding-energy estimates. We fit one ridge predictor using the Track A features and another using the expanded group. The later incremental comparison tests whether embeddings add predictive value above each control group.

The table below gives each scalar feature's range and Spearman rank association with expression. The codon block supplies 64 predictor columns and is excluded from this one-value-per-feature summary. Association describes how measured values vary together; held-out prediction is evaluated next.


In [4]:
# Membership is explicit: adding a diagnostic control must not change these groups.
track_a_members = (
    "GC3",
    "codon use (64)",
    "Nussinov, first 200 nt",
)
control_groups = {
    "Track A controls": track_a_members,
    "all controls": (
        "GC",
        *track_a_members,
        "MFE, first 60 nt",
        "MFE, whole sequence",
    ),
}

if READ_SAVED_RESULTS:
    CONTROL_SETS = dict.fromkeys(control_groups)
    print("Feature-expression diagnostic skipped; saved prediction summaries remain available below.")
else:
    # Join columns in the stated order, keeping every matrix aligned to the same rows.
    CONTROL_SETS = {
        name: np.hstack([CONTROLS[member] for member in members])
        for name, members in control_groups.items()
    }

    # Describe association with expression; no prediction model is fitted here.
    display(pd.DataFrame({
        "Spearman with expression": [
            spearmanr(FEATURES[c], y).statistic for c in FEATURES
        ],
        "min": FEATURES.min(),
        "max": FEATURES.max(),
    }).round(3))


,Spearman with expression,min,max
GC,-0.173,0.445,0.546
GC3,-0.249,0.376,0.677
"Nussinov, first 200 nt",0.132,0.650,0.810
"MFE, first 60 nt",0.264,-21.800,-2.800
"MFE, whole sequence",-0.144,-229.400,-155.900


The first-60-nucleotide folding energy and GC3 have the largest absolute rank associations in this table, with opposite signs. These associations do not establish causation or show that the features supply distinct information from one another. They also do not establish that the folding predictions match cellular RNA structures.

We now have individual controls, two explicitly defined control groups and the saved embeddings. The next comparison asks how well each predicts expression on rows held out from fitting.


---

## Compare separate predictors

Fit a separate ridge predictor to each encoder's pooled embeddings, each individual control, each combined control set and a random-feature reference. Each input matrix has one row per sampled sequence; its columns are the features available to that predictor. The pretrained encoders remain frozen.

Protein encoders provide a reference without sequence input that distinguishes these synonymous variants. The fixed random-feature matrix supplies another reference, with 640 Gaussian columns. It is one width and one random draw: assignment seeds change the folds, not these features. The cell below collects these comparisons before any fitting.

Read mode collects only these comparison names. It does not reconstruct feature matrices or generate a new random-feature reference.


In [ ]:
if READ_SAVED_RESULTS:
    # Only comparison names are needed to inspect the requested assignment keys.
    MODELS = dict.fromkeys((
        *LABEL.values(), *CONTROLS, *CONTROL_SETS, "random features (640)",
    ))
else:
    # Fix the random-feature reference once; assignment seeds below only change folds.
    rng = np.random.default_rng(0)
    NULL = rng.normal(size=(len(seqs), 640))

    # Every candidate matrix follows the same row order as expression y and groups.
    MODELS = {
        **{LABEL[k]: X for k, X in EMBEDDINGS.items()},
        **CONTROLS,
        **CONTROL_SETS,
        "random features (640)": NULL,
    }

KIND = {
    **{LABEL[k]: MODALITY[k] for k in LABEL},
    **{c: "control" for c in (*CONTROLS, *CONTROL_SETS)},
    "random features (640)": "null",
}
SEEDS = (PROBE_SEED, *range(N_ASSIGNMENTS))

if len(SEEDS) != len(set(SEEDS)):
    raise ValueError("The pinned and further assignment seeds must be distinct.")


### Evaluate on grouped folds

Each fold assignment divides the **1,000 sampled rows** into five folds, keeping identical sequences together. The predictor fits on four folds and is evaluated on the fifth. This repeats until every fold has served as the held-out fold. Feature scaling and predictor fitting use the outer training rows, and expression is scored on the held-out rows. Stage 1's pinned assignment uses seed 42; the ten further assignments use seeds 0 through 9.

The [linear-probing guide](../../docs/methods/sequences-to-predictions.md#linear-probing) explains scaling, ridge penalties and the current limitation of internal penalty selection. In read mode, a coverage table checks the actual model/seed keys, and only complete requested groups enter the historical summary. Missing or ambiguous assignments are reported, not fitted. Calculation mode additionally checks fingerprints before computing and saving missing compatible assignments. Changing the seed list cannot make an unrelated same-sized set appear complete; other saved rows remain untouched.


In [ ]:
if READ_SAVED_RESULTS:
    probe_groups = {name: [(name, seed) for seed in SEEDS] for name in MODELS}
    probe_rows, probe_coverage = mrfp_cache.complete_groups(
        SAVED_TABLES[PROBE_CACHE], probe_groups,
    )
    scores = pd.DataFrame(probe_rows, columns=["model", "seed", "r2"])
    display(pd.DataFrame(probe_coverage))
    print("Only complete requested model/seed groups enter the historical summary; nothing missing is fitted.")


In [ ]:
if not READ_SAVED_RESULTS:
    # Validate every requested model/seed identity before calculating any missing score.
    probe_inputs = {
        name: mrfp_cache.fingerprint(
            {"sample": sample_identity, "model": name, "metric": "mean outer-fold R2"},
            arrays=(X,),
            functions=(analysis.probe_scores, analysis._ridge_pipeline, grouped_folds),
            packages=("numpy", "scipy", "scikit-learn"),
        )
        for name, X in MODELS.items()
    }
    probe_expected = {
        (name, seed): mrfp_cache.fingerprint({"inputs": identity, "seed": seed})
        for name, identity in probe_inputs.items()
        for seed in SEEDS
    }
    probe_keys, probe_numeric = CACHE_FIELDS[PROBE_CACHE]
    scores = pd.read_csv(PROBE_CACHE) if PROBE_CACHE.exists() else pd.DataFrame()
    saved = mrfp_cache.checked_rows(
        scores.to_dict("records"), keys=probe_keys, expected=probe_expected,
        numeric=probe_numeric, source=PROBE_CACHE,
    )

    for name, X in MODELS.items():
        missing = [seed for seed in SEEDS if (name, seed) not in saved]
        if not missing:
            continue
        # Keep the same five-fold calculation; only missing exact assignment keys run.
        rows = [
            {
                "model": name, "seed": seed,
                "r2": analysis.probe_scores(X, y, groups=groups, seed=seed).mean(),
                mrfp_cache.IDENTITY_COLUMN: probe_expected[name, seed],
            }
            for seed in missing
        ]
        records = mrfp_cache.upsert(
            scores.to_dict("records"), rows,
            keys=probe_keys, expected=probe_expected, numeric=probe_numeric, source=PROBE_CACHE,
        )
        scores = pd.DataFrame(records)
        save(scores, PROBE_CACHE)

    # Extra historical seed keys stay on disk but do not enter the requested summary.
    saved = mrfp_cache.checked_rows(
        scores.to_dict("records"), keys=probe_keys, expected=probe_expected,
        numeric=probe_numeric, source=PROBE_CACHE,
    )
    scores = pd.DataFrame([saved[key] for key in probe_expected])


### Read the individual prediction scores

The prediction score for a held-out fold is:

$$
R^2 = 1 - \frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}.
$$

In this notation:

- $i$ indexes the rows in that held-out fold.
- $y_i$ is the supplied log corrected-fluorescence target and $\hat y_i$ is its prediction on that same scale.
- $\bar y$ is the mean of those log-scale targets within that held-out fold.
- Each sum adds squared deviations over those same rows.

A score of zero matches the squared error of predicting that fold's mean log corrected fluorescence; a negative score has larger error than that reference. Predictions are not exponentiated before scoring, so $R^2$ describes errors on the log scale and is not the fraction of protein molecules predicted correctly.

For example, a prediction corresponding to twice the observed corrected fluorescence has an error magnitude of $\ln(2)\approx0.693$, whether the observed corrected value is 100 or 10,000. A prediction corresponding to half the observed value has the opposite log error and the same squared error. This makes the numerator sensitive to fold errors instead of raw fluorescence differences; the denominator still compares those errors with the variation in that fold's log-scale targets. The [target guide](../../docs/methods/data-and-targets.md#gfpuv-normalization-and-the-natural-logarithm) connects this scale to the original measurements. `analysis.probe_scores` returns five fold scores, and the notebook averages them for each assignment.

**Pinned** uses Stage 1's fixed assignment. The remaining score columns give the mean and standard deviation of the five-fold means over ten further assignments. That spread describes sensitivity to partitioning the same sample; it is not a confidence interval or variation across independent datasets.


In [5]:
if scores.empty:
    print("No complete saved probe groups are available; prediction summary skipped.")
else:
    # Separate the pinned result from variation across the further assignments.
    pinned = scores[scores["seed"] == PROBE_SEED].set_index("model")["r2"]
    further = scores[scores["seed"] != PROBE_SEED].groupby("model")["r2"]
    PROBES = pd.DataFrame({
        "kind": pd.Series(KIND),
        "R2, pinned": pinned,
        f"R2, {N_ASSIGNMENTS} assignments": further.mean(),
        "SD over assignments": further.std(ddof=0),
    }).loc[[name for name in MODELS if name in set(scores["model"])]]
    display(PROBES.round(3))


,kind,"R2, pinned","R2, 10 assignments",SD over assignments
NT 500M human-ref,DNA,0.296,0.291,0.013
NT v2 100M multi-species,DNA,0.281,0.264,0.016
DNABERT-2,DNA,0.226,0.206,0.012
HyenaDNA large,DNA,0.249,0.247,0.021
RNA-FM,RNA,0.222,0.223,0.008
RiNALMo 150M,RNA,0.285,0.284,0.019
mRNA-FM,RNA,0.463,0.466,0.013
CaLM,RNA,0.298,0.304,0.021
ESM-2 8M,protein,-0.003,-0.005,0.002
ESM-2 35M,protein,-0.003,-0.005,0.002


The controls predict some variation in log corrected fluorescence: codon frequencies alone average approximately $R^2=0.184$, and the first-60-nucleotide folding energy scores $0.108$. The Track A group scores $0.190$, rising to $0.306$ when GC and both energies are added. This does not isolate which added feature accounts for the combined improvement or establish an optimal folding window.

Among the pooled representations, mRNA-FM has the highest mean over the ten further assignments, approximately $R^2=0.466$, followed by CaLM at $0.304$ and NT 500M at $0.291$. Protein predictors and the random-feature reference have slightly negative mean scores, consistent with their roles as references without useful variant-specific sequence input.

These separate scores leave an important question open: do the embeddings improve predictions beyond what the controls already capture? A strong individual score alone cannot answer that. The next comparison fits an additional predictor to errors left by the controls.


---

## Test what the embeddings add above the controls

Retain the same rows and control sets, and add an embedding-based predictor to one already using the controls. `C` contains control features, `X` contains one encoder's embeddings or the random features, and `y` contains the same log corrected-fluorescence targets; all three follow the same sampled row order.

**Construct the additional predictor's target.** Within each outer training subset, make control predictions through an inner set of grouped folds. For each training row, subtract the prediction made without fitting on its sequence group:

$$
r_i = y_i - \hat y_{i,\mathrm{inner}}.
$$

In this notation:

- $i$ identifies an outer-training row and $y_i$ is its supplied log corrected-fluorescence target.
- $\hat y_{i,\mathrm{inner}}$ is the control prediction from an inner fold that held out that row's sequence group.
- $r_i$ is the residual error on this log scale, used to train the predictor on `X`.

This gives the additional predictor a target based on held-out control errors. The outer test rows remain outside both training procedures.

**Evaluate the combined prediction.** Refit the control predictor on all outer-training rows. Train the additional predictor using encoder embeddings (or the fixed random features) as inputs and the previously calculated, held-out control errors as targets. Those residual targets come from the earlier inner-fold predictions. Use both fitted predictors to predict the outer test rows, add their predictions, and compare the combined score with the controls alone:

$$
\hat y_{j,\mathrm{combined}} = \hat y_{j,\mathrm{controls}} + \hat r_j,
\qquad
\Delta R^2 = R^2_{\mathrm{combined}} - R^2_{\mathrm{controls}}.
$$

In this notation:

- $j$ identifies an outer-test row.
- $\hat y_{j,\mathrm{controls}}$ is the control predictor's expression prediction, and $\hat r_j$ is the additional predictor's estimated residual.
- $\hat y_{j,\mathrm{combined}}$ is their sum.
- The two $R^2$ scores use the same outer-test rows; $\Delta R^2$ is their difference.

The two predictors fit their penalties separately. This evaluates a particular residual-prediction procedure, which differs from fitting a single ridge model to concatenated controls and embeddings. The explicitly grouped outer and residual-prediction folds do not change `RidgeCV`'s internal penalty-selection defaults; the [evaluation limitation](../../docs/methods/sequences-to-predictions.md#current-evaluation-limitation) still applies.


In [ ]:
def ridge():
    # Use a fresh scaler and model for each independent fit, retaining the same grid.
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 5, 20))),
    ])


def gain_over(C, X, seed):
    """Held-out R^2 of the controls alone and of the controls plus X, on the same folds."""
    base, stacked = [], []
    for train, test in grouped_folds(groups, seed=seed):
        # Inner indices refer to the outer-training subset, not the complete dataset.
        inner_groups = [groups[i] for i in train]
        inner = grouped_folds(inner_groups, seed=f"inner-{seed}")

        # Form training residuals from control predictions held out by sequence group.
        honest = cross_val_predict(ridge(), C[train], y[train], cv=inner)

        # Predict the same test rows with controls and a model trained on residual targets.
        controls = ridge().fit(C[train], y[train]).predict(C[test])
        extra = ridge().fit(X[train], y[train] - honest).predict(X[test])
        base.append(r2_score(y[test], controls))
        stacked.append(r2_score(y[test], controls + extra))

    # Each returned value is an unweighted mean over the five outer folds.
    return float(np.mean(base)), float(np.mean(stacked))


### Evaluate the paired comparison

Apply this procedure to every encoder and the same random-feature reference, above each of the two control groups. Both sides of each comparison use the same test rows. Across the ten assignments, only the fold partition changes; the sampled rows, feature definitions and residual-prediction method stay fixed.

Read mode reports coverage for each candidate/control pair and summarizes only pairs with every requested assignment. It neither fits residual predictors nor fills missing seeds. Calculation mode checks identities before fitting missing assignments and uses exact-key updates rather than appending duplicate results. Unknown or conflicting identities stop that calculation path; historical display remains available in read mode.


In [ ]:
if READ_SAVED_RESULTS:
    gain_groups = {
        (set_name, name): [(set_name, name, seed) for seed in range(N_ASSIGNMENTS)]
        for set_name in CONTROL_SETS
        for name in (*LABEL.values(), "random features (640)")
    }
    gain_rows, gain_coverage = mrfp_cache.complete_groups(
        SAVED_TABLES[GAIN_CACHE], gain_groups,
    )
    gains = pd.DataFrame(
        gain_rows,
        columns=["controls", "model", "seed", "controls R2", "with encoder R2"],
    )
    display(pd.DataFrame(gain_coverage))
    print("Only complete requested control/model/seed groups enter the historical summary; no residual models are fitted.")


In [ ]:
if not READ_SAVED_RESULTS:
    # Compare each encoder, plus the same random reference, above each control set.
    CANDIDATES = {
        **{LABEL[k]: X for k, X in EMBEDDINGS.items()},
        "random features (640)": NULL,
    }
    gain_inputs = {
        (set_name, name): mrfp_cache.fingerprint(
            {"sample": sample_identity, "controls": set_name, "model": name},
            arrays=(C, X),
            functions=(gain_over, ridge, grouped_folds),
            packages=("numpy", "scipy", "scikit-learn"),
        )
        for set_name, C in CONTROL_SETS.items()
        for name, X in CANDIDATES.items()
    }
    gain_expected = {
        (set_name, name, seed): mrfp_cache.fingerprint({"inputs": identity, "seed": seed})
        for (set_name, name), identity in gain_inputs.items()
        for seed in range(N_ASSIGNMENTS)
    }
    gain_keys, gain_numeric = CACHE_FIELDS[GAIN_CACHE]
    gains = pd.read_csv(GAIN_CACHE) if GAIN_CACHE.exists() else pd.DataFrame()
    saved = mrfp_cache.checked_rows(
        gains.to_dict("records"), keys=gain_keys, expected=gain_expected,
        numeric=gain_numeric, source=GAIN_CACHE,
    )

    for set_name, C in CONTROL_SETS.items():
        for name, X in CANDIDATES.items():
            missing = [
                seed for seed in range(N_ASSIGNMENTS)
                if (set_name, name, seed) not in saved
            ]
            if not missing:
                continue
            rows = []
            for seed in missing:
                base, stacked = gain_over(C, X, seed)
                rows.append({
                    "model": name, "controls": set_name, "seed": seed,
                    "controls R2": base, "with encoder R2": stacked,
                    mrfp_cache.IDENTITY_COLUMN: gain_expected[set_name, name, seed],
                })
            # Exact-key upsert prevents reruns from appending duplicate assignments.
            records = mrfp_cache.upsert(
                gains.to_dict("records"), rows,
                keys=gain_keys, expected=gain_expected, numeric=gain_numeric, source=GAIN_CACHE,
            )
            gains = pd.DataFrame(records)
            save(gains, GAIN_CACHE)

    saved = mrfp_cache.checked_rows(
        gains.to_dict("records"), keys=gain_keys, expected=gain_expected,
        numeric=gain_numeric, source=GAIN_CACHE,
    )
    gains = pd.DataFrame([saved[key] for key in gain_expected])


### Read the added predictive value

`gain_over` averages the five outer-fold scores for each assignment. Within each fold assignment, subtract the controls-only mean $R^2$ from the combined prediction's mean $R^2$. This produces ten gains, one per assignment. The table reports their average (**gain**), their **lowest** and **highest** values, and how many are positive (**assignments above 0**). A positive gain supports predictive usefulness beyond the selected controls under this procedure; it does not identify a biological mechanism or exhaust the possible sequence controls.


In [6]:
if gains.empty:
    print("No complete saved paired comparisons are available; added-value summary skipped.")
else:
    # Subtract paired assignment means before summarizing their variation.
    gains["gain"] = gains["with encoder R2"] - gains["controls R2"]
    GAINS = (
        gains.groupby(["controls", "model"], sort=False)
        .agg(**{
            "controls R2": ("controls R2", "mean"),
            "with encoder R2": ("with encoder R2", "mean"),
            "gain": ("gain", "mean"),
            "lowest": ("gain", "min"),
            "highest": ("gain", "max"),
            "assignments above 0": ("gain", lambda g: int((g > 0).sum())),
        })
    )
    GAINS.insert(0, "kind", [KIND[m] for _, m in GAINS.index])
    display(GAINS.round(3))


kind  controls R2  with encoder R2   gain  lowest  highest  assignments above 0
controls         model                                                                                                       
Track A controls NT 500M human-ref             DNA        0.190            0.320  0.130   0.118    0.146                   10
                 NT v2 100M multi-species      DNA        0.190            0.251  0.061   0.025    0.079                   10
                 DNABERT-2                     DNA        0.190            0.233  0.044   0.028    0.058                   10
                 HyenaDNA large                DNA        0.190            0.266  0.077   0.048    0.098                   10
                 RNA-FM                        RNA        0.190            0.244  0.054   0.041    0.065                   10
                 RiNALMo 150M                  RNA        0.190            0.287  0.098   0.068    0.120                   10
                 mRNA-FM                       RNA        0.190            0.451  0.261   0.231    0.290                   10
                 CaLM                          RNA        0.190            0.282  0.093   0.064    0.121                   10
                 ESM-2 8M                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.190            0.188 -0.001  -0.003    0.000                    1
all controls     NT 500M human-ref             DNA        0.306            0.381  0.076   0.067    0.088                   10
                 NT v2 100M multi-species      DNA        0.306            0.334  0.029   0.010    0.048                   10
                 DNABERT-2                     DNA        0.306            0.332  0.026   0.010    0.034                   10
                 HyenaDNA large                DNA        0.306            0.342  0.036   0.005    0.057                   10
                 RNA-FM                        RNA        0.306            0.327  0.021   0.011    0.030                   10
                 RiNALMo 150M                  RNA        0.306            0.355  0.049   0.033    0.067                   10
                 mRNA-FM                       RNA        0.306            0.488  0.182   0.168    0.203                   10
                 CaLM                          RNA        0.306            0.328  0.022  -0.001    0.043                    9
                 ESM-2 8M                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.306            0.305 -0.001  -0.004    0.001                    2

In the retained results, every nucleotide encoder has positive gain above Track A in all ten assignments. Above all controls, each still has a positive mean gain; seven gain in every assignment, while CaLM gains in nine. Protein gains are negligible at the displayed precision, and the random reference has a slightly negative mean gain.

mRNA-FM has the largest mean gain above all controls: the combined prediction scores approximately $R^2=0.488$, compared with $0.306$ for controls alone, giving a gain of $0.182$. NT 500M follows with a gain of about $0.076$. This gain compares the residual-based combination with controls alone; it is not the difference between the individual-predictor scores in the previous table.

The result answers the opening question for this procedure: the selected controls leave expression variation that the nucleotide embeddings help predict. It does not identify a causal mechanism or establish that all composition and RNA-structure information has been accounted for.


---

## Findings, limitations and connection to Stage 4

For this sample, nucleotide representations add predictive usefulness beyond the selected sequence and folding controls under the specified ridge procedure. The controls themselves are informative, and mRNA-FM gives the strongest results in both the individual and incremental comparisons. The protein reference behaves as expected when synonymous variants share the same amino-acid input.

This evidence concerns a sample of synonymous variants of one gene, pooled embeddings and the chosen linear predictors. Repeated fold assignments repartition the same examples; they do not establish performance on independent genes or datasets. The gains do not identify a biological mechanism, validate the predicted RNA structures inside cells or rule out nonlinear relationships involving the same controls. The [internal penalty-selection limitation](../../docs/methods/sequences-to-predictions.md#current-evaluation-limitation) also remains part of the evaluation.

As a Stage 1 representation control, this establishes why the information that varies between examples matters before selecting a fusion method. The [encoder-swap experiments](../stage4/encoder_swaps.ipynb) ask whether promising representations remain useful with token-level prediction heads and whether combining them adds value beyond a validation-selected single encoder. A strong nucleotide result here motivates that comparison; it does not predict a fusion gain by itself. Generalizing the explanation requires a stated expectation for a new task or input condition and an evaluation that could contradict it.

The [mRFP run record and its prose qualification](../../docs/run-records.md#2026-10-19-stage4-mrfp-control-the-mrfp-synonymous-control-research-plan-item-5) retain the execution account and earlier interpretation. This notebook's discussion draws on those existing results.
